# Carnatify — CNN raga classifier (Colab GPU)

Trains a CNN on **tonic-normalized log-CQT** spectrograms of Demucs-separated vocals.

Key design choices (differ from the original brief — see repo discussion):
- **Log-CQT + row-roll tonic normalization** instead of pitch-shifting audio to 220 Hz: in log-frequency a pitch shift is an exact bin shift, so normalization is free and lossless. Sa lands on a fixed bin for every track.
- **Track-grouped train/val split** — segments of one recording never straddle the split (segment-level splits leak and inflate accuracy).
- **Track-level evaluation** (mean softmax over a track's segments), compared against the honest live-pipeline RF baseline: **40.5% top-1 / 60.8% top-3**, not the 77.8% CompMusic CV number.
- All tonics are precomputed (annotated or already-estimated) — shipped in `manifest.json`.

**Needs on Drive:** `carnatify_concert_audio.zip` (already there) + `cnn_extra_audio.zip` (from `build_cnn_extra_audio.py`).
Runtime → GPU. Every stage is resumable. Total ~3-4h first run.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
%pip -q install demucs soundfile librosa

In [ ]:
!unzip -q -n /content/drive/MyDrive/carnatify_concert_audio.zip -d /content/
!unzip -q -n /content/drive/MyDrive/cnn_extra_audio.zip -d /content/
import json
MANIFEST = json.load(open('/content/cnn_extra_audio/manifest.json'))
print(len(MANIFEST['segments']), 'local segments;', len(MANIFEST['archive_tonics']), 'archive tonics')

In [ ]:
# ── Stage 1: cut all sources into 65s stereo wavs for Demucs ──────────────
import re
import numpy as np, librosa, soundfile as sf
from pathlib import Path

SEGS = Path('/content/segs'); SEGS.mkdir(exist_ok=True)
SR = 44100; SEG_LEN = 65.0
EXCLUDED = {'Rāgamālika'}
meta = {}  # seg stem -> (track_id, raga, tonic, source)

def write_seg(y, stem):
    out = SEGS / f'{stem}.wav'
    if not out.exists():
        sf.write(out, np.repeat(y[:, None], 2, axis=1), SR)  # true stereo: Demucs mono-expand crashes on torch>=2

# local (saraga + varnam) — already cut, just convert flac -> stereo wav
for s in MANIFEST['segments']:
    stem = re.sub(r'[^\w\-]', '_', s['file'].replace('/', '__').replace('.flac', ''))
    meta[stem] = (s['track_id'], s['raga'], float(s['tonic']), s['source'])
    if not (SEGS / f'{stem}.wav').exists():
        y, _ = librosa.load(f"/content/cnn_extra_audio/{s['file']}", sr=SR, mono=True)
        write_seg(y, stem)
print('local segs staged:', len(meta))

# archive — cut 3 evenly-placed 65s segments per track, tonic from manifest
for raga_dir in sorted(Path('/content/concert_audio').iterdir()):
    if not raga_dir.is_dir() or raga_dir.name in EXCLUDED:
        continue
    for mp3 in sorted(raga_dir.glob('*.mp3')):
        tid = f'archive__{raga_dir.name}__{mp3.stem}'
        info = MANIFEST['archive_tonics'].get(tid)
        if info is None:
            continue
        try:
            dur = librosa.get_duration(path=str(mp3))
        except Exception:
            continue
        if dur < 50:
            continue
        n = min(3, max(1, int((dur - 30) // SEG_LEN)))
        usable = dur - 30 - SEG_LEN
        starts = [30 + usable * i / max(1, n - 1) for i in range(n)] if n > 1 else [min(30, max(0, dur - SEG_LEN))]
        base = re.sub(r'[^\w\-]', '_', tid)
        for i, st in enumerate(starts):
            stem = f'{base}__{i}'
            meta[stem] = (tid, info['raga'], float(info['tonic']), 'archive')
            if (SEGS / f'{stem}.wav').exists():
                continue
            y, _ = librosa.load(str(mp3), sr=SR, mono=True, offset=max(0, st), duration=SEG_LEN)
            if len(y) >= SR * 50:
                write_seg(y, stem)
            else:
                meta.pop(stem)
print('total segments:', len(meta))
json.dump({k: list(v) for k, v in meta.items()}, open('/content/seg_meta.json', 'w'))

In [ ]:
# ── Stage 2: Demucs vocal separation on GPU, batched, resumable ───────────
import subprocess
from pathlib import Path

OUT = Path('/content/demucs_out')
todo = [p for p in sorted(Path('/content/segs').glob('*.wav'))
        if not (OUT / 'htdemucs' / p.stem / 'vocals.wav').exists()]
print(len(todo), 'segments need separation')
for i in range(0, len(todo), 24):
    batch = [str(p) for p in todo[i:i+24]]
    r = subprocess.run(['python', '-m', 'demucs', '--two-stems=vocals', '-n', 'htdemucs',
                        '-d', 'cuda', '-o', str(OUT)] + batch, capture_output=True, text=True)
    if r.returncode != 0:
        print('batch failed:', r.stderr[-300:])
    print(f'{min(i+24, len(todo))}/{len(todo)}', flush=True)
print('separated:', len(list(OUT.rglob('vocals.wav'))))

In [ ]:
# ── Stage 3: tonic-normalized log-CQT slices -> Drive (resumable) ─────────
import json
import numpy as np, librosa
from pathlib import Path

meta = {k: tuple(v) for k, v in json.load(open('/content/seg_meta.json')).items()}
FEATS = Path('/content/drive/MyDrive/carnatify_cnn_feats'); FEATS.mkdir(exist_ok=True)
SR22, HOP, BPO, NBINS = 22050, 512, 36, 180
FMIN = librosa.note_to_hz('C2')
WIN, WHOP = 215, 215  # 5s slices, no overlap (overlap = near-duplicates)
SA_BIN = 60

done = {p.stem for p in FEATS.glob('*.npz')}
todo = [k for k in meta if k not in done]
print(len(todo), 'segments need features')
for n, stem in enumerate(sorted(todo), 1):
    voc = Path('/content/demucs_out/htdemucs') / stem / 'vocals.wav'
    if not voc.exists():
        continue
    tid, raga, tonic, source = meta[stem]
    y, _ = librosa.load(voc, sr=SR22, mono=True)
    C = np.abs(librosa.cqt(y, sr=SR22, hop_length=HOP, fmin=FMIN,
                           n_bins=NBINS, bins_per_octave=BPO))
    logC = librosa.amplitude_to_db(C, ref=np.max)
    roll = int(round(SA_BIN - BPO * np.log2(tonic / FMIN)))
    logC = np.roll(logC, roll, axis=0)
    slices = [logC[:, s:s+WIN] for s in range(0, logC.shape[1] - WIN + 1, WHOP)]
    if not slices:
        continue
    X = np.stack(slices).astype(np.float16)
    np.savez_compressed(FEATS / f'{stem}.npz', X=X, track_id=tid, raga=raga, source=source)
    if n % 50 == 0:
        print(f'{n}/{len(todo)}', flush=True)
print('feature files on Drive:', len(list(FEATS.glob('*.npz'))))

In [ ]:
# ── Stage 4: dataset assembly + track-grouped split ───────────────────────
import unicodedata, json
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict

def fold(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s.lower()) if not unicodedata.combining(c)).strip()

FEATS = Path('/content/drive/MyDrive/carnatify_cnn_feats')
X_all, y_all, g_all = [], [], []
for p in sorted(FEATS.glob('*.npz')):
    d = np.load(p, allow_pickle=True)
    raga = fold(str(d['raga']))
    tid = str(d['track_id'])
    for x in d['X']:
        X_all.append(x); y_all.append(raga); g_all.append(tid)

tracks_per_raga = {r: len({g for g, y in zip(g_all, y_all) if y == r}) for r in set(y_all)}
keep = {r for r, n in tracks_per_raga.items() if n >= 5}
print(f'{len(keep)} ragas with >=5 tracks (dropping {len(tracks_per_raga)-len(keep)})')

mask = [y in keep for y in y_all]
X_all = np.stack([x for x, m in zip(X_all, mask) if m])
y_all = np.array([y for y, m in zip(y_all, mask) if m])
g_all = np.array([g for g, m in zip(g_all, mask) if m])
ragas = sorted(keep)
y_idx = np.array([ragas.index(y) for y in y_all])

# hold out ~20% of TRACKS per raga
rng = np.random.default_rng(42)
val_tracks = set()
for r in ragas:
    tr = sorted({g for g, y in zip(g_all, y_all) if y == r})
    rng.shuffle(tr)
    val_tracks.update(tr[:max(1, round(0.2 * len(tr)))])
is_val = np.array([g in val_tracks for g in g_all])
print(f'slices: {len(y_idx)} train={np.sum(~is_val)} val={np.sum(is_val)}; '
      f'tracks: train={len(set(g_all[~is_val]))} val={len(val_tracks)}')

In [ ]:
# ── Stage 5a: model/dataset defs + sanity overfit check ───────────────────
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class RagaCNN(nn.Module):
    def __init__(self, n_classes, p_conv=0.1, p_fc=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(p_conv),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(p_conv),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)), nn.Dropout2d(p_conv),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(p_fc),
            nn.Linear(256, n_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

class SliceDS(Dataset):
    def __init__(self, X, y, train):
        self.X, self.y, self.train = X, y, train
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        x = self.X[i].astype(np.float32)
        if self.train:
            x = x.copy()
            t = np.random.randint(0, x.shape[1] // 4)          # SpecAugment time mask
            t0 = np.random.randint(0, x.shape[1] - t) if t else 0
            x[:, t0:t0+t] = x.mean()
            f = np.random.randint(0, x.shape[0] // 16)         # freq mask, mild: rows ARE pitch classes here
            f0 = np.random.randint(0, x.shape[0] - f) if f else 0
            x[f0:f0+f, :] = x.mean()
            x = x + np.random.uniform(-3, 3)                    # gain
            x = np.roll(x, np.random.randint(-3, 4), axis=0)    # tonic-error jitter (±1 semitone)
        x = (x + 40.0) / 40.0  # dB in [-80,0] -> roughly [-1,1]
        return torch.from_numpy(x[None]), self.y[i]

dev = 'cuda'

# Sanity: memorize 512 unaugmented train slices with zero dropout.
# ~90%+ by epoch 40 => pipeline/model fine, main-run failure = regularization/data.
# Stuck low => input or label wiring broken; stop and report.
sub = np.random.default_rng(0).choice(np.where(~is_val)[0], 512, replace=False)
sm = RagaCNN(len(ragas), p_conv=0.0, p_fc=0.0).to(dev)
sdl = DataLoader(SliceDS(X_all[sub], y_idx[sub], train=False), batch_size=64, shuffle=True)
sopt = torch.optim.Adam(sm.parameters(), lr=1e-3)
sce = nn.CrossEntropyLoss()
for ep in range(40):
    sm.train(); tot = corr = 0
    for xb, yb in sdl:
        xb, yb = xb.to(dev), yb.to(dev)
        sopt.zero_grad(); out = sm(xb)
        sce(out, yb).backward(); sopt.step()
        tot += len(yb); corr += (out.argmax(1) == yb).sum().item()
    if (ep + 1) % 5 == 0:
        print(f'sanity overfit epoch {ep+1:2d}: train acc={corr/tot:.3f}', flush=True)
del sm, sdl, sopt
torch.cuda.empty_cache()


In [ ]:
# ── Stage 5b: main training (lighter regularization than first run) ───────
model = RagaCNN(len(ragas), p_conv=0.1, p_fc=0.3).to(dev)
counts = Counter(y_idx[~is_val].tolist())
w = torch.tensor([1.0 / counts.get(i, 1) for i in range(len(ragas))], dtype=torch.float32)
crit = nn.CrossEntropyLoss(weight=(w / w.sum() * len(ragas)).to(dev))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 60
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
tr_dl = DataLoader(SliceDS(X_all[~is_val], y_idx[~is_val], True), batch_size=64, shuffle=True, num_workers=2)
va_dl = DataLoader(SliceDS(X_all[is_val], y_idx[is_val], False), batch_size=64, num_workers=2)
val_gids = g_all[is_val]  # aligned with va_dl order (no shuffle)

def track_level_eval():
    model.eval()
    probs = defaultdict(list)
    sl_tot = sl_corr = 0
    with torch.no_grad():
        i = 0
        for xb, yb in va_dl:
            p = torch.softmax(model(xb.to(dev)), 1).cpu().numpy()
            sl_corr += int((p.argmax(1) == yb.numpy()).sum()); sl_tot += len(yb)
            for row in p:
                probs[val_gids[i]].append(row); i += 1
    t1 = t3 = 0
    for tid, ps in probs.items():
        mean = np.mean(ps, 0)
        true = y_idx[g_all == tid][0]
        order = np.argsort(-mean)
        t1 += order[0] == true
        t3 += true in order[:3]
    return t1 / len(probs), t3 / len(probs), sl_corr / sl_tot, len(probs)

best = 0.0
for epoch in range(EPOCHS):
    model.train()
    tot = correct = 0
    for xb, yb in tr_dl:
        xb, yb = xb.to(dev), yb.to(dev)
        opt.zero_grad()
        out = model(xb)
        loss = crit(out, yb)
        loss.backward(); opt.step()
        tot += len(yb); correct += (out.argmax(1) == yb).sum().item()
    sched.step()
    t1, t3, sl, ntr = track_level_eval()
    marker = ''
    if t1 > best:
        best = t1
        torch.save({'model_state': model.state_dict(), 'ragas': ragas,
                    'val_top1': t1, 'val_top3': t3,
                    'config': {'sr': 22050, 'hop': 512, 'bpo': 36, 'n_bins': 180,
                               'fmin_note': 'C2', 'sa_bin': 60, 'win': 215}},
                   '/content/drive/MyDrive/carnatify_cnn_model.pt')
        marker = '  <- saved'
    print(f'epoch {epoch+1:2d}  train_slice_acc={correct/tot:.3f}  val_slice_acc={sl:.3f}  '
          f'VAL track top1={t1:.3f} top3={t3:.3f} (n={ntr}){marker}', flush=True)
print(f'\nbest val track top1={best:.1%} — RF live baseline: 40.5% top1 / 60.8% top3')
